In [1]:
import pandas as pd
import duckdb

In [2]:
con = duckdb.connect()

con.execute("""
CREATE OR REPLACE TABLE paysim AS
SELECT *
FROM read_csv_auto('../data/paysim_transactions.csv');
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
row_count = con.execute("""
SELECT COUNT(*) AS total_rows
FROM paysim;
""").df()

row_count

,total_rows
0,6362620


In [4]:
con.execute("SELECT 50000 as threshold uniON ALL SELECT 100000 as threshold UNION ALL SELECT 200000 as threshold union all SELECT 500000 as threshold union all select 1000000 as threshold").df()

,threshold
0,50000
1,100000
2,200000
3,500000
4,1000000


In [14]:
con.execute("""
WITH thresholds AS (
    SELECT 50000 AS threshold
    UNION ALL 
    SELECT 100000
    UNION ALL 
    SELECT 200000
    UNION ALL 
    SELECT 500000
    UNION ALL 
    SELECT 1000000
),

rule_flags AS (
    SELECT
        th.threshold AS rule_threshold,
        t.type,
        t.amount,
        t.isFraud,
        CASE
            WHEN t.type IN ('TRANSFER', 'CASH_OUT')
                 AND t.amount >= th.threshold
            THEN 1
            ELSE 0
        END AS rule_flag
    FROM paysim t
    CROSS JOIN thresholds th
)

SELECT *
FROM rule_flags
LIMIT 20;
""").df()

,rule_threshold,type,amount,isFraud,rule_flag
0,1000000,PAYMENT,9839.64,0,0
1,1000000,PAYMENT,1864.28,0,0
2,1000000,TRANSFER,181.00,1,0
3,1000000,CASH_OUT,181.00,1,0
4,1000000,PAYMENT,11668.14,0,0
5,1000000,PAYMENT,7817.71,0,0
6,1000000,PAYMENT,7107.77,0,0
7,1000000,PAYMENT,7861.64,0,0
8,1000000,PAYMENT,4024.36,0,0
9,1000000,DEBIT,5337.77,0,0


In [15]:
con.execute("""
WITH thresholds AS (
    SELECT 50000 AS threshold
    UNION ALL 
    SELECT 100000
    UNION ALL 
    SELECT 200000
    UNION ALL 
    SELECT 500000
    UNION ALL 
    SELECT 1000000
),

rule_flags AS (
    SELECT
        th.threshold AS rule_threshold,
        t.type,
        t.amount,
        t.isFraud,
        CASE
            WHEN t.type IN ('TRANSFER', 'CASH_OUT')
                 AND t.amount >= th.threshold
            THEN 1
            ELSE 0
        END AS rule_flag
    FROM paysim t
    CROSS JOIN thresholds th
)

SELECT
    rule_threshold,
    COUNT(*) AS total_rule_tests,
    SUM(rule_flag) AS flagged_transactions,
    SUM(isFraud) AS total_fraud_transactions,
    SUM(CASE WHEN rule_flag = 1 AND isFraud = 1 THEN 1 ELSE 0 END) AS true_positives,
    SUM(CASE WHEN rule_flag = 1 AND isFraud = 0 THEN 1 ELSE 0 END) AS false_positives,
    SUM(CASE WHEN rule_flag = 0 AND isFraud = 1 THEN 1 ELSE 0 END) AS false_negatives
FROM rule_flags
GROUP BY rule_threshold
ORDER BY rule_threshold;
""").df()

,rule_threshold,total_rule_tests,flagged_transactions,total_fraud_transactions,true_positives,false_positives,false_negatives
0,50000,6362620,2359396.0,8213.0,7175.0,2352221.0,1038.0
1,100000,6362620,1934297.0,8213.0,6506.0,1927791.0,1707.0
2,200000,6362620,1197669.0,8213.0,5471.0,1192198.0,2742.0
3,500000,6362620,314301.0,8213.0,3864.0,310437.0,4349.0
4,1000000,6362620,130507.0,8213.0,2706.0,127801.0,5507.0


In [16]:
con.execute("""
WITH thresholds AS (
    SELECT 50000 AS threshold
    UNION ALL 
    SELECT 100000
    UNION ALL 
    SELECT 200000
    UNION ALL 
    SELECT 500000
    UNION ALL 
    SELECT 1000000
),

rule_flags AS (
    SELECT
        th.threshold AS rule_threshold,
        t.type,
        t.amount,
        t.isFraud,
        CASE
            WHEN t.type IN ('TRANSFER', 'CASH_OUT')
                 AND t.amount >= th.threshold
            THEN 1
            ELSE 0
        END AS rule_flag
    FROM paysim t
    CROSS JOIN thresholds th
),

rule_summary AS (
    SELECT
        rule_threshold,
        COUNT(*) AS total_rule_tests,
        SUM(rule_flag) AS flagged_transactions,
        SUM(isFraud) AS total_fraud_transactions,
        SUM(CASE WHEN rule_flag = 1 AND isFraud = 1 THEN 1 ELSE 0 END) AS true_positives,
        SUM(CASE WHEN rule_flag = 1 AND isFraud = 0 THEN 1 ELSE 0 END) AS false_positives,
        SUM(CASE WHEN rule_flag = 0 AND isFraud = 1 THEN 1 ELSE 0 END) AS false_negatives
    FROM rule_flags
    GROUP BY rule_threshold
)

SELECT
    rule_threshold,
    flagged_transactions,
    total_fraud_transactions,
    true_positives,
    false_positives,
    false_negatives,
    ROUND(true_positives * 1.0 / total_fraud_transactions, 4) AS recall,
    ROUND(true_positives * 1.0 / flagged_transactions, 4) AS precision
FROM rule_summary
ORDER BY rule_threshold;
""").df()

,rule_threshold,flagged_transactions,total_fraud_transactions,true_positives,false_positives,false_negatives,recall,precision
0,50000,2359396.0,8213.0,7175.0,2352221.0,1038.0,0.8736,0.0030
1,100000,1934297.0,8213.0,6506.0,1927791.0,1707.0,0.7922,0.0034
2,200000,1197669.0,8213.0,5471.0,1192198.0,2742.0,0.6661,0.0046
3,500000,314301.0,8213.0,3864.0,310437.0,4349.0,0.4705,0.0123
4,1000000,130507.0,8213.0,2706.0,127801.0,5507.0,0.3295,0.0207


## Threshold Tuning Interpretation

As the amount threshold increases, flagged transactions and false positives decrease significantly, while recall also decreases.

Precision improves as the threshold becomes stricter, but remains low overall. This suggests that transaction amount helps reduce alert volume, but amount alone is not sufficient to create a high-quality fraud detection rule.

The 200,000 threshold captures more fraud but creates excessive false positives. The 500,000 threshold significantly reduces alert volume and improves precision, but misses more fraud.

Based on the current rule, 500,000 may be a more practical baseline for workload control, while further rule improvements are needed to increase precision without losing excessive recall.

In [19]:
threshold_results = con.execute("""
WITH thresholds AS (
    SELECT 50000 AS threshold
    UNION ALL 
    SELECT 100000
    UNION ALL 
    SELECT 200000
    UNION ALL 
    SELECT 500000
    UNION ALL 
    SELECT 1000000
),

rule_flags AS (
    SELECT
        th.threshold AS rule_threshold,
        t.type,
        t.amount,
        t.isFraud,
        CASE
            WHEN t.type IN ('TRANSFER', 'CASH_OUT')
                 AND t.amount >= th.threshold
            THEN 1
            ELSE 0
        END AS rule_flag
    FROM paysim t
    CROSS JOIN thresholds th
),

rule_summary AS (
    SELECT
        rule_threshold,
        COUNT(*) AS total_rule_tests,
        SUM(rule_flag) AS flagged_transactions,
        SUM(isFraud) AS total_fraud_transactions,
        SUM(CASE WHEN rule_flag = 1 AND isFraud = 1 THEN 1 ELSE 0 END) AS true_positives,
        SUM(CASE WHEN rule_flag = 1 AND isFraud = 0 THEN 1 ELSE 0 END) AS false_positives,
        SUM(CASE WHEN rule_flag = 0 AND isFraud = 1 THEN 1 ELSE 0 END) AS false_negatives
    FROM rule_flags
    GROUP BY rule_threshold
)

SELECT
    rule_threshold,
    flagged_transactions,
    total_fraud_transactions,
    true_positives,
    false_positives,
    false_negatives,
    ROUND(true_positives * 1.0 / total_fraud_transactions, 4) AS recall,
    ROUND(true_positives * 1.0 / flagged_transactions, 4) AS precision
FROM rule_summary
ORDER BY rule_threshold;
""").df()

threshold_results.to_csv("../outputs/threshold_tuning_results.csv", index=False)

threshold_results

,rule_threshold,flagged_transactions,total_fraud_transactions,true_positives,false_positives,false_negatives,recall,precision
0,50000,2359396.0,8213.0,7175.0,2352221.0,1038.0,0.8736,0.0030
1,100000,1934297.0,8213.0,6506.0,1927791.0,1707.0,0.7922,0.0034
2,200000,1197669.0,8213.0,5471.0,1192198.0,2742.0,0.6661,0.0046
3,500000,314301.0,8213.0,3864.0,310437.0,4349.0,0.4705,0.0123
4,1000000,130507.0,8213.0,2706.0,127801.0,5507.0,0.3295,0.0207
